## **OSEMN | Obtain: Data Acquisition | CDS Link using Web Scraping**
To obtain CDS data, web scraping approach is initially implemented using the BeautifulSoup library in Python. BeautifulSoup is employed to parse HTML content from institutional websites and extract relevant data fields from publicly available CDS pages.

**1. Install and Import Libraries**

In [ ]:
!pip install beautifulsoup4 requests pandas lxml openpyxl

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
import time

**2. Create Institution Data Input File**

An input file was prepared containing the list of 60 selected institutions and their corresponding website URLs. This is to enable the data webscraping process across all institutions for the CDS dataset.

File path: [/content/drive/MyDrive/P2 MDS/01_Data_Acquisition/01_data_input/cds_input.xlsx](https://docs.google.com/spreadsheets/d/1DO6Q9i-ZfS2Gk6r0i8qF8LyEa6gGwsJH/edit?usp=share_link&ouid=118039523433753400814&rtpof=true&sd=true)

**3. Environment Setup: Mount Drive to upload in Google Drive**

The input file is uploaded in Google Drive. Google Drive is mounted to acces the files and save the extracted files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
input_file = "/content/drive/MyDrive/P2 MDS/01_Data_Acquisition/01_data_input/cds_input.xlsx"
df = pd.read_excel(input_file)
df.head()

,Institution,Website_url,Found CDS Link
0,Harvard University,https://oir.harvard.edu,NaN
1,Yale University,https://oir.yale.edu,NaN
2,Princeton University,https://oir.princeton.edu,NaN
3,Columbia University,https://irp.columbia.edu,NaN
4,Brown University,https://oir.brown.edu,NaN


**4. Find Institutions CDS Links to extract the pdf or webpage for the CDS dataset**

In [ ]:
def find_cds_links(page_url):
  #set up request headers and try accessing the webpage
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    #if the request fails, returns empty list and an error message
    try:
        response = requests.get(page_url, headers=headers, timeout=20)
        response.raise_for_status()
    except Exception as e:
        return [], f"request failed: {e}"

    #parse the html content of the page using beautifulsoup
    soup = BeautifulSoup(response.text, "html.parser")
    links_found = []

    #define keywords to identify common data set links
    keywords = [
        "common data set",
        "cds"
    ]

    year_pattern = re.compile(r'20(1[9]|2[0-9])')

    #loop through all links on the page
    for a in soup.find_all("a", href=True):
        href = a.get("href", "").strip()
        text = a.get_text(" ", strip=True).lower()
        full_url = urljoin(page_url, href)

        combined_text = f"{text} {href.lower()}" # combine the link text and its href for comprehensive keyword searching

        #check if the link is related to CDS
        if any(keyword in combined_text for keyword in keywords):

            #determine file type and check if a year exists
            file_type = "PDF" if ".pdf" in href.lower() else "Webpage"
            year_match = year_pattern.search(combined_text)
            year_found = year_match.group(0) if year_match else None

            #save link details into the list
            links_found.append({
                "link_text": a.get_text(" ", strip=True),
                "cds_url": full_url,
                "file_type": file_type,
                "year_found": year_found
            })

    #return all matching links found
    return links_found, "Success"

**5. Initial Webscraping Results**

In [ ]:
#initialize an empty list to store scraping results
results = []

#iterate through each row (institution) in the dataframe
for idx, row in df.iterrows():
    institution = row["Institution"]
    website_url = row["Website_url"]

    #print progress
    print(f"Checking {idx+1}/{len(df)}: {institution}")

    #find cds links on the webpage
    cds_links, status = find_cds_links(website_url)

    #if cds links are found save the file
    if cds_links:
        for item in cds_links:
            results.append({
                "institution": institution,
                "source_page": website_url,
                "status": status,
                "link_text": item["link_text"],
                "cds_url": item["cds_url"],
                "file_type": item["file_type"],
                "year_found": item["year_found"]
            })

    # if no cds links are found, save empty record
    else:
        results.append({
            "institution": institution,
            "source_page": website_url,
            "status": status if status != "Success" else "no cds link found",
            "link_text": None,
            "cds_url": None,
            "file_type": None,
            "year_found": None
        })

    #pause for 2 secs before the next request
    time.sleep(2)

Checking 1/60: Harvard University
Checking 2/60: Yale University
Checking 3/60: Princeton University
Checking 4/60: Columbia University
Checking 5/60: Brown University
Checking 6/60: Dartmouth College
Checking 7/60: Cornell University
Checking 8/60: University of Pennsylvania
Checking 9/60: Stanford University
Checking 10/60: Massachusetts Institute of Technology (MIT)
Checking 11/60: University of Chicago
Checking 12/60: Duke University
Checking 13/60: Northwestern University
Checking 14/60: Johns Hopkins University
Checking 15/60: California Institute of Technology (Caltech)
Checking 16/60: University of Michigan – Ann Arbor
Checking 17/60: University of Virginia
Checking 18/60: University of North Carolina – Chapel Hill
Checking 19/60: University of California – Berkeley
Checking 20/60: University of California – Los Angeles (UCLA)
Checking 21/60: University of California – San Diego (UC San Diego)
Checking 22/60: University of California – Davis (UC Davis)
Checking 23/60: Universit

**6. Results**

In [ ]:
results_df = pd.DataFrame(results)
results_df.head()

,institution,source_page,status,link_text,cds_url,file_type,year_found
0,Harvard University,https://oir.harvard.edu,request failed: HTTPSConnectionPool(host='oir....,None,None,None,None
1,Yale University,https://oir.yale.edu,Success,Common Data Set,https://oir.yale.edu/common-data-set,Webpage,None
2,Yale University,https://oir.yale.edu,Success,Common Data Set (CDS):,https://oir.yale.edu/common-data-set,Webpage,None
3,Princeton University,https://oir.princeton.edu,request failed: HTTPSConnectionPool(host='oir....,None,None,None,None
4,Columbia University,https://irp.columbia.edu,request failed: HTTPSConnectionPool(host='irp....,None,None,None,None


In [ ]:
output_file = "content/drive/MyDrive/P2 MDS/01_Data_Acquisition/01_data_input/v1_cdslink_input.xlsx"
results_df.to_excel(output_file, index=False)
print(f"Saved to: {output_file}")

Saved to: /content/drive/MyDrive/P2 MDS/P2 MDS | Final/Step1 | CDS Extraction/v1_cdslink_input.xlsx


**6.1 Extraction Summary | Initial**

In [ ]:
print(results_df["status"].value_counts())

status
Success                                                                                                                                                                                                                                                                                           34
no cds link found                                                                                                                                                                                                                                                                                 22
request failed: HTTPSConnectionPool(host='oir.princeton.edu', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x78a684ad5a60>: Failed to resolve 'oir.princeton.edu' ([Errno -2] Name or service not known)"))           1
request failed: HTTPSConnectionPool(host='oir.harvard.edu', port=443): Read timed out. (read timeout=20)          

**6.2 Overall Extraction Success Rate | Initial**

The initial success rate is calculated based on the number of extracted records marked as successful.

In [ ]:
total = len(results_df)
found = len(results_df[results_df["status"] == "Success"])

success_rate = (found / total) * 100

print(f"Total institutions: {total}")
print(f"CDS links found: {found}")
print(f"Success Rate: {success_rate:.2f}%")

Total institutions: 78
CDS links found: 34
Success Rate: 43.59%


**6.3 Institutions Level Success Rate | Initial**

The success rate is recalculated at the institution level to avoid overstating performance when one institution produces multiple CDS related links.

In [ ]:
total_unique_institutions = results_df['institution'].nunique()
unique_successful_institutions = results_df[results_df["status"] == "Success"]['institution'].nunique()

success_rate = (unique_successful_institutions / total_unique_institutions) * 100

print(f"Total unique institutions: {total_unique_institutions}")
print(f"Unique institutions with CDS links found: {unique_successful_institutions}")
print(f"Success Rate (unique institutions): {success_rate:.2f}%")

Total unique institutions: 60
Unique institutions with CDS links found: 16
Success Rate (unique institutions): 26.67%


**7. Improve CDS Link Findings**

The initial scraping approach failed when institutiions placed the CDS links on internal pages such as institutional research, facts and figures, or planning pages. An improved search strategy is implemented to identify more reliable CDS links.

**7.1 Reload Institution Website Dataset**

Institution website dataset is reloaded before applying the improved CDS discovery method.

In [ ]:
input_file = "/content/drive/MyDrive/P2 MDS/01_Data_Acquisition/01_data_input/cds_input.xlsx"
df = pd.read_excel(input_file)
df.head()

,Institution,Website_url,Found CDS Link
0,Harvard University,https://oir.harvard.edu,NaN
1,Yale University,https://oir.yale.edu,NaN
2,Princeton University,https://oir.princeton.edu,NaN
3,Columbia University,https://irp.columbia.edu,NaN
4,Brown University,https://oir.brown.edu,NaN


**7.2 Helper Functions**

Helper functions created to improved scraping process

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0"}

#send a request to the webpage safely
def safe_get(url, timeout=20):
    try:
        r = requests.get(url, headers=HEADERS, timeout=timeout)
        r.raise_for_status()
        return r
    except Exception:
        return None

#check if two URLs belong to the same domain
def same_domain(url1, url2):
    try:
        from urllib.parse import urlparse
        d1 = urlparse(url1).netloc.replace("www.", "")
        d2 = urlparse(url2).netloc.replace("www.", "")
        return d1 == d2
    except:
        return False

#check and standardize text with single space, strip and lowercase
def clean_text(text):
    return re.sub(r"\s+", " ", str(text)).strip().lower()

**7.3 Page Link Generation**

A list of likely CDS page paths generated for each institution. The scraper check common locations where universities often publish CDS files such as institutional research, facts, data or planning pages.

In [ ]:
def generate_candidate_pages(base_url):
    base_url = base_url.rstrip("/")

    candidate_paths = [
        "",
        "/common-data-set",
        "/common_data_set",
        "/common-data",
        "/cds",
        "/institutional-research",
        "/institutional_research",
        "/office-of-institutional-research",
        "/institutionalresearch",
        "/research",
        "/facts",
        "/fact-book",
        "/factbook",
        "/about/facts",
        "/about",
        "/ir",
        "/data",
        "/planning",
        "/about-us/facts-and-figures"
    ]

    return [base_url + path for path in candidate_paths]

**7.4 CDS Link Function**

Check hyperlinks if it is a valid CDS link. Calculate a score to estimate how likely a link is a cds link

In [ ]:
def score_link(link_text, href):
    text = clean_text(link_text)
    href_lower = clean_text(href)
    combined = text + " " + href_lower

    #start with score of 0
    score = 0

    #add points for keywords commonly found in cds links
    if "common data set" in combined: # high score for exact phrase match
        score += 12
    if "common-data-set" in combined or "common_data_set" in combined: # good score for hyphenated/underscored phrase
        score += 10
    if re.search(r"\bcds\b", combined): # moderate score for 'cds' as a whole word
        score += 8
    if ".pdf" in href_lower: # higher score if it's a PDF link
        score += 5
    if re.search(r"20(1[9]|2[0-9])", combined): # score for recent years (2019-2029)
        score += 3
    if "institutional research" in combined: # score for institutional research keywords
        score += 2
    if "fact book" in combined or "facts" in combined: # score for fact book/facts keywords
        score += 1

    #terms that are usually unrelated to CDS page
    bad_terms = [
        "admissions",
        "apply",
        "calendar",
        "course",
        "library",
        "news",
        "events",
        "facebook",
        "instagram",
        "linkedin",
        "youtube"
    ]

    #reduce score if unrelated terms appear
    for term in bad_terms:
        if term in combined:
            score -= 3

    #return finalscore
    return score # return the calculated score

**7.5 CDS Link Generation Extraction**

In [ ]:
def extract_candidate_links(page_url):

    #retrieve webpage content and returns empty list if it fails
    response = safe_get(page_url)
    if response is None:
        return []

    #parse HTML content and initialise links
    soup = BeautifulSoup(response.text, "html.parser")
    candidates = []

    #iterate anchor tags containing href attributes
    for a in soup.find_all("a", href=True):

        #extract hyperlink reference and visible anchor text
        href = a.get("href", "").strip()
        text = a.get_text(" ", strip=True)

        #ignore links with empty href
        if not href:
            continue

        #converts URL
        full_url = urljoin(page_url, href)

        #ignore non http links
        if not full_url.startswith("http"):
            continue

        #evaluation
        score = score_link(text, full_url)

        #stores links
        if score > 0:
            candidates.append({
                "source_page": page_url,
                "link_text": text,
                "found_link": full_url,
                "score": score
            })

    #returns links
    return candidates

**7.6 Best CDS Link Selection**

In [ ]:
def find_best_cds_link(website_url, max_internal_pages=5):

    #initialize a list to store all candidate links
    all_candidates = []
    candidate_pages = generate_candidate_pages(website_url)
    checked_pages = set()

    #pass1 - check likely direct paths
    for page in candidate_pages:
        if page in checked_pages:
            continue
        checked_pages.add(page)

        links = extract_candidate_links(page)
        all_candidates.extend(links)

        time.sleep(1)

    #pass2 - follow promising internal pages from homepage
    homepage_response = safe_get(website_url)
    if homepage_response is not None:
        soup = BeautifulSoup(homepage_response.text, "html.parser")
        promising_pages = []

        keywords = [
            "institutional research",
            "common data set",
            "cds",
            "facts",
            "fact book",
            "data",
            "research",
            "planning"
        ]

        #extract relevant internal links
        for a in soup.find_all("a", href=True):
            href = a.get("href", "").strip()
            text = clean_text(a.get_text(" ", strip=True))
            full_url = urljoin(website_url, href)

            #exclude invalid links
            if not full_url.startswith("http"):
                continue
            if not same_domain(website_url, full_url):
                continue

            #combine link text and URL
            combined = text + " " + clean_text(full_url)

            if any(keyword in combined for keyword in keywords):
                promising_pages.append(full_url)

        #remove duplicates and limit pages
        unique_promising_pages = []
        for p in promising_pages:
            if p not in unique_promising_pages:
                unique_promising_pages.append(p)

        #analyse
        for page in unique_promising_pages[:max_internal_pages]:
            if page in checked_pages:
                continue
            checked_pages.add(page)

            #extract and store links when it is valid
            links = extract_candidate_links(page)
            all_candidates.extend(links)

            time.sleep(1)

    #no links identified
    if not all_candidates:
        return None

    #rank
    all_candidates = sorted(all_candidates, key=lambda x: x["score"], reverse=True)

    #return ranked link
    return all_candidates[0]

**7.7 Improved Institution CDS Link Extraction**

In [ ]:
#initialize an empty list to store the results
results = []

#iterate through each row (institution) in the DataFrame
for idx, row in df.iterrows():
    institution = str(row["Institution"]).strip()
    website_url = str(row["Website_url"]).strip()

    #print current progress
    print(f"Checking {idx+1}/{len(df)}: {institution}")

    #check if website URL is missing or 'nan'
    if not website_url or website_url.lower() == "nan":
        results.append({
            "Institution": institution,
            "Website URL": None,
            "Found CDS Link": None,
            "Source Page": None,
            "Link Text": None,
            "Score": None,
            "Status": "Missing Website URL"
        })
        continue

    #identify best match
    best_match = find_best_cds_link(website_url)

    #store if valid link is found
    if best_match:
        results.append({
            "Institution": institution,
            "Website URL": website_url,
            "Found CDS Link": best_match["found_link"],
            "Source Page": best_match["source_page"],
            "Link Text": best_match["link_text"],
            "Score": best_match["score"],
            "Status": "Found"
        })

    #no CDS link is identified
    else:
        results.append({
            "Institution": institution,
            "Website URL": website_url,
            "Found CDS Link": None,
            "Source Page": None,
            "Link Text": None,
            "Score": None,
            "Status": "Check manually"
        })

    #pause for 2 secs before the next request
    time.sleep(2)

results_df = pd.DataFrame(results)
results_df.head()

Checking 1/60: Harvard University
Checking 2/60: Yale University
Checking 3/60: Princeton University
Checking 4/60: Columbia University
Checking 5/60: Brown University
Checking 6/60: Dartmouth College
Checking 7/60: Cornell University
Checking 8/60: University of Pennsylvania
Checking 9/60: Stanford University
Checking 10/60: Massachusetts Institute of Technology (MIT)
Checking 11/60: University of Chicago
Checking 12/60: Duke University
Checking 13/60: Northwestern University
Checking 14/60: Johns Hopkins University
Checking 15/60: California Institute of Technology (Caltech)
Checking 16/60: University of Michigan – Ann Arbor
Checking 17/60: University of Virginia
Checking 18/60: University of North Carolina – Chapel Hill
Checking 19/60: University of California – Berkeley
Checking 20/60: University of California – Los Angeles (UCLA)
Checking 21/60: University of California – San Diego (UC San Diego)
Checking 22/60: University of California – Davis (UC Davis)
Checking 23/60: Universit

,Institution,Website URL,Found CDS Link,Source Page,Link Text,Score,Status
0,Harvard University,https://oir.harvard.edu,None,None,None,NaN,Check manually
1,Yale University,https://oir.yale.edu,https://oir.yale.edu/common-data-set,https://oir.yale.edu,Common Data Set (CDS):,30.0,Found
2,Princeton University,https://oir.princeton.edu,None,None,None,NaN,Check manually
3,Columbia University,https://irp.columbia.edu,None,None,None,NaN,Check manually
4,Brown University,https://oir.brown.edu,https://oir.brown.edu/institutional-data/commo...,https://oir.brown.edu,Common Data Set,22.0,Found


**7.8 Extraction Results | Improved**

In [ ]:
results_df = pd.DataFrame(results)
results_df.head()

,Institution,Website URL,Found CDS Link,Source Page,Link Text,Score,Status
0,Harvard University,https://oir.harvard.edu,None,None,None,NaN,Check manually
1,Yale University,https://oir.yale.edu,https://oir.yale.edu/common-data-set,https://oir.yale.edu,Common Data Set (CDS):,30.0,Found
2,Princeton University,https://oir.princeton.edu,None,None,None,NaN,Check manually
3,Columbia University,https://irp.columbia.edu,None,None,None,NaN,Check manually
4,Brown University,https://oir.brown.edu,https://oir.brown.edu/institutional-data/commo...,https://oir.brown.edu,Common Data Set,22.0,Found


**7.9 Overall Extraction Summary | Improved**

In [ ]:
output_file = "/content/drive/MyDrive/P2 MDS/01_Data_Acquisition/02_data_scraping_output/v1_cdslink_input_improved.xlsx"
results_df.to_excel(output_file, index=False)
print(f"Saved to: {output_file}")

Saved to: /content/drive/MyDrive/P2 MDS/P2 MDS | Final/Step1 | CDS Extraction/v1_cdslink_input_improved.xlsx


**7.10 Overall Extraction Success Rate | Improved**

The final success rate is calculated based on the how many institutions produced CDS link using the improved method

In [ ]:
print(results_df["Status"].value_counts())

Status
Found             34
Check manually    26
Name: count, dtype: int64


In [ ]:
total_unique_institutions = results_df['Institution'].nunique()
unique_successful_institutions = results_df[results_df["Status"] == "Found"]['Institution'].nunique()

success_rate = (unique_successful_institutions / total_unique_institutions) * 100

print(f"Total unique institutions: {total_unique_institutions}")
print(f"Unique institutions with CDS links found: {unique_successful_institutions}")
print(f"Success Rate (unique institutions): {success_rate:.2f}%")

Total unique institutions: 60
Unique institutions with CDS links found: 34
Success Rate (unique institutions): 56.67%
